# 04 — Prebuilt Receipt: Validate Expense Receipts for Claims

**Time**: ~5 minutes · **Model**: `prebuilt-receipt` · **No training required**

---

## Insurance Scenario

A policyholder submits **expense receipts** — hotel stays after an accident, pharmacy purchases, car rental charges — as part of a claim. You need to automatically extract merchant, date, items, and total for reimbursement validation.

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult
import pandas as pd

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Analyze a Receipt

In [ ]:
receipt_url = "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/sdk/documentintelligence/azure-ai-documentintelligence/samples/sample_forms/receipt/contoso-receipt.png"

poller = client.begin_analyze_document(
    "prebuilt-receipt",
    AnalyzeDocumentRequest(url_source=receipt_url)
)
result: AnalyzeResult = poller.result()

print(f"Receipts found: {len(result.documents) if result.documents else 0}")

## Extract Receipt Details

In [ ]:
def format_currency(field):
    """Format a currency field."""
    if field and field.get("valueCurrency"):
        curr = field["valueCurrency"]
        return f"{curr.get('currencyCode', '')} {curr.get('amount', 'N/A')}"
    return field.get("content", "N/A") if field else "N/A"

if result.documents:
    receipt = result.documents[0]
    f = receipt.fields
    
    print("=" * 50)
    print("RECEIPT DETAILS")
    print("=" * 50)
    print(f"  Receipt type:     {receipt.doc_type or 'N/A'}")
    print(f"  Merchant:         {f.get('MerchantName', {}).get('content', 'N/A')}")
    print(f"  Address:          {f.get('MerchantAddress', {}).get('content', 'N/A')}")
    print(f"  Phone:            {f.get('MerchantPhoneNumber', {}).get('content', 'N/A')}")
    print(f"  Date:             {f.get('TransactionDate', {}).get('content', 'N/A')}")
    print(f"  Time:             {f.get('TransactionTime', {}).get('content', 'N/A')}")
    print(f"  Subtotal:         {format_currency(f.get('Subtotal'))}")
    print(f"  Tax:              {format_currency(f.get('TotalTax'))}")
    print(f"  Tip:              {format_currency(f.get('Tip'))}")
    print(f"  TOTAL:            {format_currency(f.get('Total'))}")

## Extract Line Items

In [ ]:
if result.documents:
    receipt = result.documents[0]
    items = receipt.fields.get("Items")
    
    if items and items.get("valueArray"):
        line_items = []
        for idx, item in enumerate(items["valueArray"]):
            obj = item.get("valueObject", {})
            line_items.append({
                "Item": obj.get("Description", {}).get("content", "N/A"),
                "Qty": obj.get("Quantity", {}).get("content", ""),
                "Price": obj.get("TotalPrice", {}).get("content", "N/A"),
            })
        
        df = pd.DataFrame(line_items)
        print("\nLINE ITEMS:")
        print(df.to_string(index=False))
    else:
        print("No line items found.")

## Build a Claim Expense Record

Transform receipt data into a structure for your expense management system.

In [ ]:
import json

if result.documents:
    receipt = result.documents[0]
    f = receipt.fields
    
    expense_record = {
        "merchant": f.get("MerchantName", {}).get("content"),
        "date": f.get("TransactionDate", {}).get("content"),
        "total": f.get("Total", {}).get("content"),
        "category": receipt.doc_type,  # e.g., "receipt.retailMeal", "receipt.hotel"
        "confidence": receipt.confidence,
        "needs_review": receipt.confidence < 0.80 if receipt.confidence else True,
    }
    
    print("Expense record for claims system:")
    print(json.dumps(expense_record, indent=2))

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Merchant extraction** | Validate that expenses are from legitimate vendors |
| **Date extraction** | Verify expenses fall within the claim period |
| **Total extraction** | Auto-calculate reimbursement amounts |
| **Receipt type** | Auto-categorize expenses (meal, hotel, retail, gas) |

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [05-prebuilt-id-document.ipynb](05-prebuilt-id-document.ipynb) | KYC: verify policyholder identity |
| [11-batch-processing.ipynb](11-batch-processing.ipynb) | Process multiple receipts at scale |